# Description

In this file, I will create the MLM dataset with 15% masking

In [1]:
import os 
import pandas as pd
from transformers import PreTrainedTokenizerFast, DataCollatorForLanguageModeling
from datasets import Dataset, DatasetDict
import numpy as np
import random
from sklearn.model_selection import train_test_split
from transformers import PreTrainedTokenizerFast, DataCollatorForLanguageModeling
from datasets import Dataset, DatasetDict
from torch.utils.data import DataLoader
import torch, random, json, os

# 1. Load data

In [2]:
PATH_DATA_FILE = os.path.join(os.getcwd(), 'dataset', 'processed', 'processed_data.csv')
PATH_TOKENIZER_FILE = os.path.join(os.getcwd(), 'python_tokenizer.json')

In [11]:
df = pd.read_csv(PATH_DATA_FILE)
list_python_function = df['method_code'].tolist()
print(f"Total functions: {len(list_python_function)}")

list_python_function = list_python_function[:10_000]  # For testing purpose, use only 1000 samples

Total functions: 1371223


In [12]:
tokenizer = PreTrainedTokenizerFast(tokenizer_file=PATH_TOKENIZER_FILE)
tokenizer.add_special_tokens({
        "pad_token": "<pad>", "unk_token": "<unk>", "mask_token": "<mask>",
        "bos_token": "<s>", "eos_token": "</s>"
    })
print("Tokenizer loaded successfully.")

Tokenizer loaded successfully.


# 2. Prepare MLM dataset

In [13]:
def prepare_mlm_dataset(functions, tokenizer, output_dir="mlm_data", mlm_prob=0.15,\
                        batch_size= 32, max_length = 512, seed=42):
    os.makedirs(output_dir, exist_ok=True)

    # 1. Split dataset (80/10/10)
    train_val, test = train_test_split(functions, test_size=0.1, random_state=42)
    train, val = train_test_split(train_val, test_size=0.1, random_state=42) 

    data = DatasetDict({
        "train": Dataset.from_dict({"text": train}),
        "validation": Dataset.from_dict({"text": val}),
        "test": Dataset.from_dict({"text": test}),
    })
    print("[DONE] Datasets split into train/val/test.")

    # 2. Tokenize function
    def tokenize_function(batch):
        return tokenizer(batch["text"], truncation=True, max_length=max_length)
    tokenize_fn = data.map(tokenize_function, batched=True, remove_columns=["text"])

    # 3. Apply random masking (MLM)
    collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=mlm_prob)

    # Convert & save each split
    for split in ["train", "validation", "test"]:
        loader = DataLoader(tokenize_fn[split], batch_size=batch_size, collate_fn=collator)
        tensors = []
        for batch in loader:
            tensors.append({
                "input_ids": batch["input_ids"],
                "labels": batch["labels"],
                "attention_mask": batch["attention_mask"]
            })
        torch.save(tensors, os.path.join(output_dir, f"{split}.pt"))
        print(f"[SAVED] {split}.pt with {len(tensors)} batches")

    print(f"[DONE] All saved in: {output_dir}")

In [14]:
prepare_mlm_dataset(list_python_function, tokenizer)

[DONE] Datasets split into train/val/test.


Map:   0%|          | 0/8100 [00:00<?, ? examples/s]

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

[SAVED] train.pt with 254 batches
[SAVED] validation.pt with 29 batches
[SAVED] test.pt with 32 batches
[DONE] All saved in: mlm_data
